# 1. 전처리 1단계: 물체 목록 만들기 (build_index)

라벨 zip(TL_*.zip)의 JSON을 읽어서 **물체 1개 = 1행**인 표(`objects_index.csv.gz`)를 만들고, 29개 라벨 코드를 **17개 클래스(판정×품목)**로 바꿉니다.

- 원본 zip은 읽기만 하고 압축을 풀지 않습니다.
- 라벨의 GPS·주소(촬영 위치)는 읽지도 저장하지도 않습니다.
- 결과: Drive `ML_TeamWork/data/processed/build_index/objects_index.csv.gz`
- 결정 근거: 프로젝트 문서 `claude/결정_기록.md`, `claude/데이터_구조_정리.md`

**작업 방법**
- 이 노트북은 **내 Drive(Colab Notebooks)에 저장해 두고 작업**합니다. Ctrl+S로 저장됩니다.
- **위에서부터 차례로 실행**하면 됩니다 (0 → 1 → 2 → 3 → 4).
- GitHub에 올리고 싶을 때만 맨 아래 **5. GitHub에 올리기** 셀을 실행합니다.

## 0. 준비 (저장소 받기·Drive 연결)
런타임이 새로 시작될 때마다 이 셀부터 실행합니다.

In [ ]:
# ── ✏️ 여기만 고치세요 ─────────────────────────────────────
BRANCH = "preprocess"                           # 전처리는 항상 이 브랜치
NEW_BRANCH = False
NOTEBOOK = "notebooks/1_preprocess/build_index.ipynb"   # GitHub에 올라갈 위치
SEED = 42
# ────────────────────────────────────────────────────────

# GitHub에서 코드를 받고(/content/26-2-1), Drive를 연결하고, 브랜치를 준비한다.
import os, subprocess, sys
REPO = "/content/26-2-1"
if not os.path.isdir(REPO):
    URL = "https://github.com/2026-2nd/26-2-1.git"
    if subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL, REPO], capture_output=True).returncode != 0:
        subprocess.run(["git", "clone", "-q", URL, REPO], check=True)   # 새 모델 브랜치면 main을 받는다
os.chdir(REPO); sys.path.insert(0, REPO)
from src.common import colab, paths
colab.setup(BRANCH, new_branch=NEW_BRANCH, notebook=NOTEBOOK)

## 1. 라벨 변환표 저장
29개 라벨 코드 → 판정(재활용/일반쓰레기/별도배출)·이유·품목·17개 클래스. 저장소의 `configs/labels_71385.json`으로 저장됩니다.

In [ ]:
%%writefile configs/labels_71385.json
{
  "_doc": "AI Hub 71385 세부 라벨(class_name) 29개 → 우리 라벨. verdict=판정, reason=일반쓰레기인 이유, item=품목(유리 색·재사용 여부와 스티로폼 색은 합침). 우리 클래스(class17) = verdict-item. 근거: claude/결정_기록.md (2026-11-26, 17개 클래스).",
  "version": "labels-v1",
  "classes": [
    "재활용-종이", "재활용-종이팩", "재활용-종이컵", "재활용-캔", "재활용-유리병", "재활용-페트", "재활용-플라스틱", "재활용-비닐",
    "일반쓰레기-종이", "일반쓰레기-종이컵", "일반쓰레기-캔", "일반쓰레기-유리병", "일반쓰레기-페트", "일반쓰레기-플라스틱", "일반쓰레기-비닐",
    "별도배출-스티로폼", "별도배출-건전지"
  ],
  "codes": {
    "c_1":          {"verdict": "재활용",     "reason": null,         "item": "종이",     "name": "종이"},
    "c_2_01":       {"verdict": "재활용",     "reason": null,         "item": "종이팩",   "name": "종이팩"},
    "c_2_02":       {"verdict": "재활용",     "reason": null,         "item": "종이컵",   "name": "종이컵"},
    "c_3":          {"verdict": "재활용",     "reason": null,         "item": "캔",       "name": "캔류"},
    "c_4_01_02":    {"verdict": "재활용",     "reason": null,         "item": "유리병",   "name": "재사용 유리(소주병+맥주병)"},
    "c_4_02_01_02": {"verdict": "재활용",     "reason": null,         "item": "유리병",   "name": "갈색 유리"},
    "c_4_02_02_02": {"verdict": "재활용",     "reason": null,         "item": "유리병",   "name": "녹색 유리"},
    "c_4_02_03_02": {"verdict": "재활용",     "reason": null,         "item": "유리병",   "name": "백색 유리"},
    "c_4_03":       {"verdict": "재활용",     "reason": null,         "item": "유리병",   "name": "기타 유리"},
    "c_5_02":       {"verdict": "재활용",     "reason": null,         "item": "페트",     "name": "페트"},
    "c_6":          {"verdict": "재활용",     "reason": null,         "item": "플라스틱", "name": "플라스틱"},
    "c_7":          {"verdict": "재활용",     "reason": null,         "item": "비닐",     "name": "비닐"},
    "c_1_01":       {"verdict": "일반쓰레기", "reason": "이물질",     "item": "종이",     "name": "종이+이물질"},
    "c_2_02_01":    {"verdict": "일반쓰레기", "reason": "이물질",     "item": "종이컵",   "name": "종이컵+이물질"},
    "c_3_01":       {"verdict": "일반쓰레기", "reason": "이물질",     "item": "캔",       "name": "캔+이물질"},
    "c_4_03_01":    {"verdict": "일반쓰레기", "reason": "이물질",     "item": "유리병",   "name": "기타유리+이물질"},
    "c_5_01_01":    {"verdict": "일반쓰레기", "reason": "이물질",     "item": "페트",     "name": "페트+이물질+다중포장재", "note": "이유가 2개라 이물질로 둠"},
    "c_5_02_01":    {"verdict": "일반쓰레기", "reason": "이물질",     "item": "페트",     "name": "페트+이물질"},
    "c_6_01":       {"verdict": "일반쓰레기", "reason": "이물질",     "item": "플라스틱", "name": "플라스틱+이물질"},
    "c_7_01":       {"verdict": "일반쓰레기", "reason": "이물질",     "item": "비닐",     "name": "비닐+이물질"},
    "c_4_01_01":    {"verdict": "일반쓰레기", "reason": "다중포장재", "item": "유리병",   "name": "재사용 유리+다중포장재"},
    "c_4_02_01_01": {"verdict": "일반쓰레기", "reason": "다중포장재", "item": "유리병",   "name": "갈색 유리+다중포장재"},
    "c_4_02_02_01": {"verdict": "일반쓰레기", "reason": "다중포장재", "item": "유리병",   "name": "녹색 유리+다중포장재"},
    "c_4_02_03_01": {"verdict": "일반쓰레기", "reason": "다중포장재", "item": "유리병",   "name": "백색 유리+다중포장재"},
    "c_5_01":       {"verdict": "일반쓰레기", "reason": "다중포장재", "item": "페트",     "name": "페트+다중포장재"},
    "c_8_01":       {"verdict": "별도배출",   "reason": null,         "item": "스티로폼", "name": "흰색 스티로폼"},
    "c_8_02":       {"verdict": "별도배출",   "reason": null,         "item": "스티로폼", "name": "컬러 스티로폼"},
    "c_8_01_01":    {"verdict": "별도배출",   "reason": null,         "item": "스티로폼", "name": "스티로폼+이물질", "note": "데이터는 별도배출로 두었지만 공식 기준으로는 일반쓰레기일 수 있음"},
    "c_9":          {"verdict": "별도배출",   "reason": null,         "item": "건전지",   "name": "건전지"}
  }
}

## 2. 물체 목록을 만드는 코드 저장
`src/preprocess/build_index.py`로 저장됩니다. 다른 노트북에서도 `from src.preprocess.build_index import build_index`로 다시 쓸 수 있습니다.

In [ ]:
%%writefile src/preprocess/build_index.py
"""라벨 zip(TL_*.zip) 안의 JSON을 읽어 '물체 1개 = 1행' 표(objects_index.csv.gz)를 만든다.

    from src.preprocess.build_index import build_index
    summary = build_index(paths.raw_dir(), out_csv, paths.configs_dir() / "labels_71385.json")

- 원본 zip은 읽기만 한다. 압축을 풀지 않는다.
- 라벨의 GPS·ADDRESS(촬영 위치)는 읽지도 저장하지도 않는다 (docs/DATA_SOURCES.md 1.3-7).
- zip 파일명의 한글은 NFD로 저장돼 있어서, 비교하기 전에 NFC로 정규화한다.
- 같은 입력이면 항상 같은 순서·같은 내용의 표가 나온다 (zip·JSON 이름 순으로 읽음).
"""
from __future__ import annotations

import csv
import gzip
import json
import shutil
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

# zip 이름에 들어 있는 수집 환경 → 짧은 코드
SOURCES = {"재활용선별장": "A", "실내형분류기": "B", "어플리케이션": "C"}

COLUMNS = [
    "object_id", "source", "env", "label_zip", "image", "obj_idx", "n_objects",
    "code", "verdict", "reason", "item", "class17", "class_id",
    "x", "y", "w", "h", "img_w", "img_h",
    "method", "date", "day", "day_night", "device", "tracking_id",
]


def nfc(text: str) -> str:
    return unicodedata.normalize("NFC", text)


def load_labels(path: str | Path) -> tuple[dict, list[str]]:
    """configs/labels_71385.json → (코드별 정보, 17개 클래스 순서)."""
    cfg = json.loads(Path(path).read_text(encoding="utf-8"))
    return cfg["codes"], cfg["classes"]


def env_of(zip_name: str) -> str:
    """'TL_실내형분류기_B1.zip' → '실내형분류기'."""
    parts = nfc(Path(zip_name).stem).split("_")
    return parts[1] if len(parts) > 1 else ""


def find_label_zips(raw_dir: str | Path, envs: list[str] | None = None) -> list[Path]:
    """raw 폴더 아래의 라벨 zip(TL_*.zip)을 찾는다. envs를 주면 그 환경만."""
    zips = [p for p in Path(raw_dir).rglob("*.zip") if nfc(p.name).startswith("TL_")]
    if envs:
        zips = [p for p in zips if env_of(p.name) in envs]
    return sorted(zips, key=lambda p: nfc(p.name))


def _resolution(text) -> tuple[str, str]:
    """'1280/960' → ('1280', '960'). 형식이 다르면 빈 값."""
    try:
        w, h = str(text).replace("*", "/").replace("x", "/").split("/")[:2]
        return str(int(float(w))), str(int(float(h)))
    except (ValueError, TypeError):
        return "", ""


def build_index(raw_dir, out_csv, labels_json, envs=None, cache_dir="/content/label_cache",
                log=print) -> dict:
    """라벨 zip을 모두 읽어 out_csv(.csv.gz)에 저장하고 요약(Counter들)을 돌려준다.

    raw_dir      paths.raw_dir()
    out_csv      저장할 파일 (예: processed/build_index/objects_index.csv.gz)
    labels_json  configs/labels_71385.json
    envs         ["실내형분류기"]처럼 일부 환경만 읽을 때. None이면 전부
    cache_dir    라벨 zip을 복사해 둘 로컬 폴더 (Drive에서 직접 읽으면 느림). None이면 복사 안 함
    """
    codes, classes = load_labels(labels_json)
    class_id = {c: i for i, c in enumerate(classes)}
    zips = find_label_zips(raw_dir, envs)
    if not zips:
        raise FileNotFoundError(f"{raw_dir} 아래에서 라벨 zip(TL_*.zip)을 찾지 못했습니다.")

    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    if cache_dir:
        Path(cache_dir).mkdir(parents=True, exist_ok=True)

    summary = {"objects": Counter(), "images": Counter(), "bad_json": Counter(),
               "unknown_code": Counter(), "no_object_image": Counter()}

    with gzip.open(out_csv, "wt", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        writer.writeheader()
        for zp in zips:
            zname = nfc(zp.name)
            env = env_of(zname)
            source = SOURCES.get(env, "?")
            src = zp
            if cache_dir:
                src = Path(cache_dir) / zname
                if not src.exists() or src.stat().st_size != zp.stat().st_size:
                    shutil.copy(zp, src)
            n_before = sum(summary["objects"].values())
            with zipfile.ZipFile(src) as zf:
                names = sorted(n for n in zf.namelist() if n.lower().endswith(".json"))
                for name in names:
                    try:
                        d = json.loads(zf.read(name).decode("utf-8-sig"))
                    except (ValueError, UnicodeDecodeError):
                        summary["bad_json"][source] += 1
                        continue
                    info = d.get("Info") or {}          # GPS·ADDRESS는 꺼내지 않는다
                    objs = d.get("objects") or []
                    image = nfc(str(d.get("Image") or Path(name).with_suffix(".jpg").name))
                    img_w, img_h = _resolution(info.get("RESOLUTION"))
                    date = str(info.get("DATE") or "")
                    summary["images"][source] += 1
                    if not objs:
                        summary["no_object_image"][source] += 1
                    for i, o in enumerate(objs):
                        code = o.get("class_name")
                        lab = codes.get(code)
                        if lab is None:
                            summary["unknown_code"][code] += 1
                            continue
                        c17 = f"{lab['verdict']}-{lab['item']}"
                        coord = (o.get("annotation") or {}).get("coord") or {}
                        writer.writerow({
                            "object_id": f"{Path(image).stem}#{i}", "source": source, "env": env,
                            "label_zip": zname, "image": image, "obj_idx": i, "n_objects": len(objs),
                            "code": code, "verdict": lab["verdict"], "reason": lab["reason"] or "",
                            "item": lab["item"], "class17": c17, "class_id": class_id[c17],
                            "x": coord.get("x"), "y": coord.get("y"),
                            "w": coord.get("width"), "h": coord.get("height"),
                            "img_w": img_w, "img_h": img_h,
                            "method": info.get("METHOD", ""), "date": date, "day": date[:10],
                            "day_night": info.get("DAY/NIGHT", ""), "device": info.get("DEVICE", ""),
                            "tracking_id": o.get("tracking_id", ""),
                        })
                        summary["objects"][source] += 1
            log(f"{zname}: 물체 {sum(summary['objects'].values()) - n_before:,}개")
    log(f"저장: {out_csv}")
    return summary

## 3. 실행 (10~20분)
zip 하나가 끝날 때마다 한 줄씩 출력됩니다. Drive에 쓸 권한이 없으면 Colab에 임시로 저장합니다.

In [ ]:
import importlib
from pathlib import Path
import src.preprocess.build_index as bi
importlib.reload(bi)

OUT = paths.processed_dir() / "build_index"      # Drive: data/processed/build_index/
try:
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / ".write_test").write_text("ok"); (OUT / ".write_test").unlink()
except OSError:
    OUT = Path("/content/processed_local/build_index"); OUT.mkdir(parents=True, exist_ok=True)
    print("⚠️ Drive에 쓸 권한이 없어 Colab에 임시 저장합니다 (런타임이 끝나면 사라짐):", OUT)

summary = bi.build_index(paths.raw_dir(), OUT / "objects_index.csv.gz",
                         paths.configs_dir() / "labels_71385.json")
print({k: dict(v) for k, v in summary.items()})

## 4. 결과 확인
다음 단계(나누는 기준 정하기)에 필요한 숫자를 봅니다: 클래스별 개수, 실내형분류기(B)의 촬영 묶음, 앱(C)의 기기 종류.

In [ ]:
import pandas as pd
df = pd.read_csv(OUT / "objects_index.csv.gz", low_memory=False)
print("물체 수:", f"{len(df):,}", "| object_id 중복 없음:", df["object_id"].is_unique)
print("\n[17개 클래스 × 출처] (C는 이미지가 없는 라벨도 포함)")
print(pd.crosstab(df["class17"], df["source"], margins=True, margins_name="합계").to_string())

b = df[df["source"] == "B"]
g = b.groupby(["method", "day"])["image"].nunique()
print(f"\n[B] 촬영 묶음(METHOD×날짜) {len(g)}개, 묶음당 이미지 수:")
print(g.describe().round(1).to_string())
print("[B] 이미지당 물체 수 평균:", round(b.groupby("image").size().mean(), 2))

c = df[df["source"] == "C"]
print(f"\n[C] 기기 {c['device'].nunique()}종, 촬영일 {c['day'].nunique()}일")
print(c.groupby("device")["image"].nunique().sort_values(ascending=False).head(15).to_string())

In [ ]:
import os, zipfile, unicodedata
import pandas as pd
from pathlib import Path
nfc = lambda s: unicodedata.normalize("NFC", s)
if "df" not in globals():
    df = pd.read_csv(OUT / "objects_index.csv.gz", low_memory=False)

# ── 실내형분류기(B) ─────────────────────────────
b = df[df["source"] == "B"].copy()
b["g_day"] = b["method"].astype(str) + "_" + b["day"].astype(str)
b["g_hour"] = b["g_day"] + "_" + b["date"].astype(str).str[11:13]

print("[B-1] 클래스마다 몇 개 묶음(분류기×날짜)에 퍼져 있나")
per = b.groupby("class17").agg(물체수=("object_id", "size"), 묶음수=("g_day", "nunique"))
per["가장큰묶음_비율"] = (b.groupby(["class17", "g_day"]).size().groupby(level=0).max() / per["물체수"]).round(2)
print(per.sort_values("묶음수").to_string())

print("\n[B-2] 묶음 개수: 날짜 단위", b["g_day"].nunique(), "/ 시간 단위", b["g_hour"].nunique())

print("\n[B-3] tracking_id가 여러 사진에 걸쳐 같은 물체를 가리키나?")
print("값 종류:", b["tracking_id"].nunique(), "| 많이 나온 값:", b["tracking_id"].value_counts().head(5).to_dict())
same = b.groupby(["g_day", "tracking_id"])["image"].nunique()
print("같은 묶음·같은 tracking_id가 나온 사진 수:", same.describe().round(1).to_dict())

# ── 앱 사진(C): 실제로 이미지가 있는 것만 ─────────────
TRAIN = next(p for p in paths.raw_dir().rglob("*") if p.is_dir() and p.name == "Training")
SRC = next(p for p in TRAIN.iterdir() if nfc(p.name).startswith("01"))
app_imgs = set()
for z in sorted(SRC.iterdir()):
    if "어플리케이션" in nfc(z.name):
        with zipfile.ZipFile(z) as zf:
            app_imgs |= {nfc(os.path.basename(n)) for n in zf.namelist() if not n.endswith("/")}
(OUT / "app_images.txt").write_text("\n".join(sorted(app_imgs)), encoding="utf-8")

c = df[(df["source"] == "C") & df["image"].isin(app_imgs)].copy()
c["g_set"] = c["device"].astype(str) + "_" + c["day"].astype(str)
print(f"\n[C-1] 이미지가 있는 앱 물체 {len(c):,}개 / 사진 {c['image'].nunique():,}장")
print(f"[C-2] 세트(폰×날짜) {c['g_set'].nunique()}개 | 폰 {c['device'].nunique()}종 | 날짜 {c['day'].nunique()}일")
print("세트당 사진 수:", c.groupby("g_set")["image"].nunique().describe().round(1).to_dict())
print("\n[C-3] 클래스마다 물체 수와 몇 개 세트에 퍼져 있나")
pc = c.groupby("class17").agg(물체수=("object_id", "size"), 세트수=("g_set", "nunique"))
print(pc.sort_values("물체수").to_string())

In [ ]:
%%writefile src/preprocess/make_split.py
"""물체 목록(objects_index)에서 학습·검증·테스트와 폰 평가 세트를 뽑는다.

    from src.preprocess.make_split import make_split
    manifest, short, info = make_split(df, app_images, seed=42)

나누는 기준 (근거: claude/결정_기록.md, 2026-11-26)
- 같은 묶음(group)의 물체는 항상 같은 세트에 들어간다. 같은 사진에서 나온 물체도 당연히 같이 간다.
    B(실내형분류기): 분류기 × 날짜 × 시간(시)  — 같은 시간대에 연달아 찍은 비슷한 사진을 묶는다
    C(앱 휴대폰)  : 폰 기종 × 날짜            — 같은 폰·같은 날 사진이 여러 용도에 섞이지 않게 한다
    A(재활용선별장): 선별장 번호(L·C·R 카메라 합침) × 날짜 × 시간
- 묶음 단위로 나누되 17개 클래스 비율이 세트마다 비슷하도록 StratifiedGroupKFold를 쓴다.
  큰 묶음 하나에 몰린 클래스가 있으면 한 번 나눈 결과가 한쪽으로 쏠릴 수 있어서,
  seed를 바꿔 가며 여러 번(TRIES) 나눠 보고 "클래스당 목표 개수에 가장 덜 모자란"(모자란 비율의 합이 가장 작은) 결과를 고른다.
  (고르는 기준은 개수뿐이고 모델 성능은 보지 않으므로 누수가 아니다. 고른 seed는 기록한다.)
- 나눈 다음, 세트마다 클래스당 최대 개수(CAPS)만큼만 무작위로 뽑는다.
- 같은 입력·같은 seed면 항상 같은 결과가 나온다 (object_id 순으로 정렬한 뒤 처리).
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 세트별 클래스당 최대 개수
B_CAPS = {"train": 1050, "val": 225, "test": 225}                         # 1,500 = 70:15:15
C_CAPS = {"phone_tune": 300, "phone_final": 500, "phone_train": 300}
A_CAPS = {"extra_a": 1050}                                                # 선별장 추가 실험용(학습에만 더함)
TRIES = 30                                                                # 나누기를 몇 번 시도해 볼지

KEEP = ["object_id", "source", "split", "class17", "class_id", "verdict", "reason", "item", "group",
        "image", "label_zip", "method", "day", "device", "x", "y", "w", "h", "img_w", "img_h"]


def add_groups(df: pd.DataFrame) -> pd.DataFrame:
    """출처별 묶음(group) 열을 붙인다."""
    df = df.copy()
    day = df["day"].astype(str)
    hour = df["date"].astype(str).str[11:13]
    method = df["method"].astype(str)
    group = pd.Series("", index=df.index, dtype=object)
    for src, key in {
        "B": "B:" + method + "_" + day + "_" + hour,
        "C": "C:" + df["device"].astype(str) + "_" + day,
        "A": "A:" + method.str.replace(r"[LCR]$", "", regex=True) + "_" + day + "_" + hour,
    }.items():
        mask = df["source"] == src
        group[mask] = key[mask]
    df["group"] = group
    return df


def _folds(sub: pd.DataFrame, n_splits: int, seed: int) -> np.ndarray:
    """묶음 단위로 n_splits개 조각에 나눈다(클래스 비율을 맞춤). 조각 번호를 돌려준다."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold = np.empty(len(sub), dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(sub, sub["class17"], sub["group"])):
        fold[idx] = k
    return fold


def _shortage(split: np.ndarray, cls: pd.Series, caps: dict) -> float:
    """세트·클래스마다 목표 개수(caps)보다 모자란 비율의 합.
    작은 세트(test 225개)에서 100개 모자란 것이 큰 세트(train 1,050개)에서 100개 모자란 것보다 더 나쁘게 계산된다."""
    counts = pd.crosstab(cls, split)
    total = 0.0
    for name, cap in caps.items():
        col = counts[name] if name in counts.columns else pd.Series(0, index=counts.index)
        total += float(((cap - col).clip(lower=0) / cap).sum())
    return round(total, 3)


def _best_split(sub: pd.DataFrame, n_splits: int, to_split, caps: dict, seed: int, tries: int):
    """seed를 바꿔 tries번 나눠 보고 가장 덜 모자란 결과를 고른다. (split 배열, 고른 seed, 모자란 합)"""
    if len(sub) == 0:
        return np.array([], dtype=object), seed, 0
    best = None
    for t in range(tries):
        split = to_split(_folds(sub, n_splits, seed + t))
        short = _shortage(split, sub["class17"], caps)
        if best is None or short < best[2]:
            best = (split, seed + t, short)
        if short == 0:
            break
    return best


def _cap(sub: pd.DataFrame, caps: dict, seed: int) -> tuple[pd.DataFrame, dict]:
    """세트·클래스마다 최대 caps[세트]개만 뽑는다. 모자라면 있는 만큼 쓰고 기록한다."""
    parts, short = [], {}
    for (split, cls), g in sub.groupby(["split", "class17"], sort=True):
        n = caps[split]
        if len(g) < n:
            short[(split, cls)] = len(g)
        parts.append(g.sample(n=min(n, len(g)), random_state=seed))
    return (pd.concat(parts) if parts else sub.iloc[0:0]), short


def make_split(df: pd.DataFrame, app_images: set[str], seed: int = 42,
               b_caps: dict = B_CAPS, c_caps: dict = C_CAPS, a_caps: dict = A_CAPS, tries: int = TRIES):
    """manifest(뽑힌 물체 목록, split 열 포함), 모자란 칸 기록(short), 고른 seed 정보(info)를 돌려준다."""
    df = add_groups(df).sort_values("object_id", kind="mergesort").reset_index(drop=True)
    out, short, info = [], {}, {}

    # B: 20조각 → 3조각 test(15%), 3조각 val(15%), 14조각 train(70%)
    b = df[df["source"] == "B"].reset_index(drop=True)
    b_split = lambda f: np.where(f < 3, "test", np.where(f < 6, "val", "train"))
    b["split"], info["B_seed"], info["B_모자란합"] = _best_split(b, 20, b_split, b_caps, seed, tries)
    picked, s = _cap(b, b_caps, seed); out.append(picked); short.update(s)

    # C: 이미지가 있는 것만. 4조각 → 1 조정용(25%), 2 최종 평가용(50%), 1 학습 추가용(25%)
    c = df[(df["source"] == "C") & df["image"].isin(app_images)].reset_index(drop=True)
    c_split = lambda f: np.select([f == 0, f <= 2], ["phone_tune", "phone_final"], "phone_train")
    c["split"], info["C_seed"], info["C_모자란합"] = _best_split(c, 4, c_split, c_caps, seed, tries)
    picked, s = _cap(c, c_caps, seed); out.append(picked); short.update(s)

    # A: 선별장은 추가 실험의 '학습에 더할 데이터'로만 쓴다
    a = df[df["source"] == "A"].reset_index(drop=True)
    a["split"] = "extra_a"
    picked, s = _cap(a, a_caps, seed); out.append(picked); short.update(s)

    manifest = pd.concat(out, ignore_index=True)
    manifest = manifest[[c for c in KEEP if c in manifest.columns]]
    manifest = manifest.sort_values(["split", "class_id", "object_id"], kind="mergesort").reset_index(drop=True)
    return manifest, short, info


def check_no_leak(manifest: pd.DataFrame) -> dict:
    """같은 묶음·같은 사진이 두 세트 이상에 들어갔는지 센다 (0이어야 정상)."""
    return {
        "여러 세트에 걸친 묶음": int((manifest.groupby("group")["split"].nunique() > 1).sum()),
        "여러 세트에 걸친 사진": int((manifest.groupby(["source", "image"])["split"].nunique() > 1).sum()),
    }

In [ ]:
import importlib
import src.preprocess.make_split as ms
importlib.reload(ms)

app_imgs = set((OUT / "app_images.txt").read_text(encoding="utf-8").split("\n"))
manifest, short, info = ms.make_split(df, app_imgs, seed=SEED)
manifest.to_csv(OUT / "split_manifest.csv.gz", index=False)
print("저장:", OUT / "split_manifest.csv.gz", f"({len(manifest):,}개)")
print("고른 seed와 모자란 정도:", info)
print("누수 검사 (0이어야 정상):", ms.check_no_leak(manifest))
print("개수가 모자란 칸 (extra_a 제외):", {k: v for k, v in short.items() if k[0] != "extra_a"} or "없음")
print("\n[클래스 × 세트]")
print(pd.crosstab(manifest["class17"], manifest["split"], margins=True, margins_name="합계").to_string())
print("\n세트별 사진 수:", manifest.groupby("split")["image"].nunique().to_dict())

In [ ]:
import shutil
from pathlib import Path
DRIVE_OUT = paths.processed_dir() / "build_index"

def can_write(d):
    try:
        d.mkdir(parents=True, exist_ok=True)
        (d / ".write_test").write_text("ok"); (d / ".write_test").unlink()
        return True
    except OSError:
        return False

if not can_write(DRIVE_OUT):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)   # 바뀐 권한을 반영하려고 다시 연결
print("Drive에 쓰기:", "가능 ✅" if can_write(DRIVE_OUT) else "아직 불가 ❌ (권한 확인 필요)")

if OUT != DRIVE_OUT:
    for f in ["objects_index.csv.gz", "app_images.txt", "split_manifest.csv.gz"]:
        shutil.copy(OUT / f, DRIVE_OUT / f)
        print("복사:", DRIVE_OUT / f)
    OUT = DRIVE_OUT

In [ ]:
for f in ["objects_index.csv.gz", "app_images.txt", "split_manifest.csv.gz"]:
    p = OUT / f
    print(f, "✅" if p.exists() else "❌ 없음", f"({p.stat().st_size/1e6:.1f}MB)" if p.exists() else "")

In [ ]:
%%writefile src/preprocess/crop.py
"""split_manifest의 물체를 원본 zip에서 꺼내 박스대로 잘라 224×224 JPEG로 저장한다.

    from src.preprocess.crop import crop_all
    crops = crop_all(manifest, paths.raw_dir(), out_dir, work_dir="/content/crops_work")

- 원본 zip은 읽기만 한다(압축을 풀지 않고 필요한 사진만 꺼낸다).
- 자르는 방법: 박스의 긴 변 × 1.1(여백 10%)인 정사각형을 박스 가운데에 맞춰 자르고 224×224로 줄인다.
  정사각형으로 자르므로 물건 모양(가로세로 비율)이 찌그러지지 않는다.
  사진 밖으로 나가는 부분은 검은색으로 채워진다(PIL 기본 동작). 모든 클래스에 똑같이 적용된다.
- 휴대폰 사진은 EXIF 회전 정보가 있을 수 있다. 라벨에 적힌 해상도(img_w×img_h)와 실제 사진 크기를 비교해
  가로세로가 뒤바뀐 경우에만 EXIF대로 돌린 뒤 자른다(orient 열에 기록).
- 큰 사진(긴 변 2,500px 이상)은 JPEG를 절반 크기로 읽어 속도를 높인다. 박스 좌표도 같은 비율로 줄인다.
- 세트(split)마다 crops/<split>.zip 하나로 묶는다. 작은 파일 수만 개를 Drive에 따로 두면 매우 느리기 때문이다.
- 이미 만든 세트 zip이 out_dir에 있으면 건너뛴다(중간에 끊겨도 이어서 할 수 있음).
- 5단계(중복 검사)에 쓰도록 잘라 낸 사진마다 dHash(64비트 사진 지문)를 계산해 둔다.
"""
from __future__ import annotations

import io
import shutil
import time
import unicodedata
import zipfile
from pathlib import Path

import pandas as pd
from PIL import Image, ImageOps

SIZE = 224
PAD = 1.10          # 여백 10%
MIN_SIDE = 32       # 원본 박스의 짧은 변이 이보다 작으면 small=True로 표시(지우지는 않음)
HALF_FROM = 2500    # 긴 변이 이 이상인 사진은 절반 크기로 읽음
QUALITY = 90


def nfc(text: str) -> str:
    return unicodedata.normalize("NFC", text)


def find_source_zips(raw_dir) -> list[Path]:
    """원천(이미지) zip(TS_*.zip) 목록."""
    return sorted((p for p in Path(raw_dir).rglob("*.zip") if nfc(p.name).startswith("TS_")),
                  key=lambda p: nfc(p.name))


def image_locations(zips: list[Path], wanted: set[str], log=print) -> dict[str, tuple[Path, zipfile.ZipInfo]]:
    """필요한 사진 이름 → (zip 경로, zip 안의 항목). zip의 파일 목록만 읽는다."""
    loc = {}
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            for info in zf.infolist():
                name = nfc(info.filename.rsplit("/", 1)[-1])
                if name in wanted and name not in loc:
                    loc[name] = (z, info)
        log(f"  목록 읽음: {nfc(z.name)} (누적 {len(loc):,}/{len(wanted):,}장)")
    return loc


def dhash(img: Image.Image, size: int = 8) -> str:
    """64비트 dHash를 16진수 16자리로."""
    g = img.convert("L").resize((size + 1, size), Image.BILINEAR)
    px = list(g.getdata())
    bits = 0
    for r in range(size):
        row = px[r * (size + 1):(r + 1) * (size + 1)]
        for c in range(size):
            bits = (bits << 1) | (row[c] > row[c + 1])
    return f"{bits:016x}"


def _open(data: bytes, img_w, img_h):
    """사진을 열고 (이미지, 박스 좌표 배율, 회전 처리 여부)를 돌려준다."""
    im = Image.open(io.BytesIO(data))
    raw_w, raw_h = im.size
    orient = "as_is"
    try:
        lw, lh = int(float(img_w)), int(float(img_h))
    except (TypeError, ValueError):
        lw = lh = None
    if lw and (lw, lh) != (raw_w, raw_h) and (lh, lw) == (raw_w, raw_h):
        im = ImageOps.exif_transpose(im)        # 라벨은 회전된 사진 기준
        orient = "exif"
    elif lw and (lw, lh) != (raw_w, raw_h):
        orient = "size_mismatch"                # 라벨 해상도와 다르면 표시만 하고 그대로 둠
    full_w = im.size[0]
    if orient == "as_is" and max(raw_w, raw_h) >= HALF_FROM:
        im.draft("RGB", (raw_w // 2, raw_h // 2))
    im = im.convert("RGB")
    return im, im.size[0] / full_w, orient


def _crop(im: Image.Image, x, y, w, h, scale: float) -> Image.Image:
    x, y, w, h = (float(v) * scale for v in (x, y, w, h))
    side = max(w, h) * PAD
    cx, cy = x + w / 2, y + h / 2
    box = (round(cx - side / 2), round(cy - side / 2), round(cx + side / 2), round(cy + side / 2))
    return im.crop(box).resize((SIZE, SIZE), Image.BICUBIC)


def crop_split(rows: pd.DataFrame, loc: dict, zip_out: Path, log=print, every: int = 2000) -> list[dict]:
    """한 세트의 물체를 잘라 zip_out에 저장하고, 물체별 기록을 돌려준다."""
    records, t0, done = [], time.time(), 0
    by_image = rows.groupby("image", sort=False)
    # zip 안에서 앞쪽에 있는 사진부터 읽는다(디스크를 덜 왔다 갔다 함)
    order = sorted(by_image.groups, key=lambda n: (str(loc[n][0]), loc[n][1].header_offset) if n in loc else ("~", 0))
    handles = {}
    with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_STORED) as out:
        for name in order:
            objs = by_image.get_group(name)
            if name not in loc:
                for _, r in objs.iterrows():
                    records.append({"object_id": r["object_id"], "path": "", "error": "원본 사진 없음"})
                continue
            zpath, info = loc[name]
            zf = handles.get(zpath) or handles.setdefault(zpath, zipfile.ZipFile(zpath))
            try:
                first = objs.iloc[0]
                im, scale, orient = _open(zf.read(info), first["img_w"], first["img_h"])
            except Exception as e:  # 깨진 사진
                for _, r in objs.iterrows():
                    records.append({"object_id": r["object_id"], "path": "", "error": f"열기 실패: {e}"[:120]})
                continue
            for _, r in objs.iterrows():
                arc = f"{r['split']}/{r['class_id']:02d}/{r['object_id'].replace('#', '_')}.jpg"
                try:
                    crop = _crop(im, r["x"], r["y"], r["w"], r["h"], scale)
                    buf = io.BytesIO(); crop.save(buf, "JPEG", quality=QUALITY)
                    out.writestr(arc, buf.getvalue())
                    records.append({"object_id": r["object_id"], "path": arc, "dhash": dhash(crop),
                                    "orient": orient, "small": min(float(r["w"]), float(r["h"])) < MIN_SIDE,
                                    "error": ""})
                except Exception as e:
                    records.append({"object_id": r["object_id"], "path": "", "error": f"자르기 실패: {e}"[:120]})
                done += 1
                if done % every == 0:
                    rate = done / (time.time() - t0)
                    log(f"    {done:,}/{len(rows):,}개 ({rate:.1f}개/초, 남은 시간 약 {(len(rows) - done) / rate / 60:.0f}분)")
    for h in handles.values():
        h.close()
    return records


def crop_all(manifest: pd.DataFrame, raw_dir, out_dir, work_dir="/content/crops_work",
             splits: list[str] | None = None, log=print) -> pd.DataFrame:
    """세트별로 잘라 out_dir/crops/<split>.zip에 저장하고, 전체 기록(crops_manifest.csv)을 돌려준다."""
    out_dir, work_dir = Path(out_dir) / "crops", Path(work_dir)
    out_dir.mkdir(parents=True, exist_ok=True); work_dir.mkdir(parents=True, exist_ok=True)
    splits = splits or sorted(manifest["split"].unique())
    todo = [s for s in splits if not (out_dir / f"{s}.zip").exists()]
    for s in sorted(set(splits) - set(todo)):
        log(f"[{s}] 이미 있음 → 건너뜀")

    if todo:
        wanted = set(manifest.loc[manifest["split"].isin(todo), "image"])
        log(f"원본 zip에서 사진 {len(wanted):,}장의 위치를 찾는 중...")
        loc = image_locations(find_source_zips(raw_dir), wanted, log)
        for s in todo:
            rows = manifest[manifest["split"] == s]
            log(f"[{s}] 물체 {len(rows):,}개 자르는 중...")
            tmp = work_dir / f"{s}.zip"
            recs = pd.DataFrame(crop_split(rows, loc, tmp, log))
            recs.to_csv(work_dir / f"{s}_records.csv", index=False)
            shutil.copy(tmp, out_dir / f"{s}.zip")
            shutil.copy(work_dir / f"{s}_records.csv", out_dir / f"{s}_records.csv")
            log(f"[{s}] 완료 → {out_dir / f'{s}.zip'} (실패 {int((recs['error'] != '').sum())}개)")

    recs = pd.concat([pd.read_csv(out_dir / f"{s}_records.csv", keep_default_na=False,
                                  dtype={"dhash": str, "path": str, "orient": str, "error": str})
                      for s in splits], ignore_index=True)
    crops = manifest.merge(recs, on="object_id", how="left")
    crops.to_csv(out_dir / "crops_manifest.csv.gz", index=False)
    log(f"저장: {out_dir / 'crops_manifest.csv.gz'}")
    return crops

In [ ]:
import importlib
import src.preprocess.crop as cr
importlib.reload(cr)

if "manifest" not in globals():
    manifest = pd.read_csv(OUT / "split_manifest.csv.gz", low_memory=False)
ORDER = ["train", "val", "test", "phone_final", "phone_tune", "phone_train", "extra_a"]
crops = cr.crop_all(manifest, paths.raw_dir(), paths.processed_dir(), splits=ORDER)

In [ ]:
import io, json, zipfile
import matplotlib.pyplot as plt
from PIL import Image

c = crops.copy()
c["ok"] = c["path"].fillna("") != ""
print("세트별 성공 개수:\n", c.groupby("split")["ok"].agg(["sum", "size"]).rename(columns={"sum": "성공", "size": "전체"}).to_string())
print("\n실패 이유:", c.loc[~c["ok"], "error"].fillna("기록 없음").str[:25].value_counts().head().to_dict())
print("회전 처리:", c["orient"].value_counts(dropna=False).to_dict())
print("작은 박스(짧은 변 32px 미만):", int(c["small"].astype(str).eq("True").sum()))

classes = json.loads((paths.configs_dir() / "labels_71385.json").read_text(encoding="utf-8"))["classes"]
sample = c[c["ok"] & (c["split"] == "train")].groupby("class_id").head(6)
fig, axes = plt.subplots(17, 6, figsize=(7, 20))
for ax in axes.flat:
    ax.axis("off")
with zipfile.ZipFile(paths.processed_dir() / "crops" / "train.zip") as z:
    for cid, g in sample.groupby("class_id"):
        for j, (_, r) in enumerate(g.iterrows()):
            axes[cid, j].imshow(Image.open(io.BytesIO(z.read(r["path"]))))
        axes[cid, 0].set_title(str(cid), loc="left", fontsize=8)
plt.tight_layout(); plt.show()
print({i: n for i, n in enumerate(classes)})

In [ ]:
%%writefile src/preprocess/check_dups.py
"""잘라 낸 사진의 지문(dHash)으로 서로 다른 세트 사이의 '거의 같은 사진'을 찾는다.

    from src.preprocess.check_dups import cross_split_dups, add_pixel_diff
    pairs = cross_split_dups(crops, [("train", "val"), ("train", "test")], max_dist=4)
    pairs = add_pixel_diff(pairs, crops_dir)      # 후보를 실제 색으로 다시 비교

- dHash는 사진을 9×8 흑백으로 줄여 옆 칸보다 밝은지 어두운지를 64비트로 적은 것이다.
  두 지문에서 다른 비트 수(해밍 거리)가 작을수록 비슷한 사진이다. 0이면 사실상 같은 사진.
- 같은 물체라도 놓인 방향이 다르면 지문이 크게 달라지므로, 이 검사는 '거의 똑같은 사진'만 잡는다.
- 실내형분류기 사진은 대부분 '어두운 배경 + 가운데 밝은 물건'이라 다른 물건도 지문이 같게 나온다.
  그래서 지문은 후보를 넓게 고르는 데만 쓰고, add_pixel_diff로 실제 색을 비교해 진짜 중복을 가린다.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

_POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def _to_u64(hexes: pd.Series) -> np.ndarray:
    return np.array([int(h, 16) for h in hexes], dtype=np.uint64)


def _hamming(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """a(n,) × b(m,) 의 해밍 거리 행렬 (n, m)."""
    x = (a[:, None] ^ b[None, :]).view(np.uint8).reshape(len(a), len(b), 8)
    return _POP[x].sum(axis=2, dtype=np.uint8)


def cross_split_dups(crops: pd.DataFrame, pairs: list[tuple[str, str]], max_dist: int = 4,
                     chunk: int = 2000) -> pd.DataFrame:
    """세트 쌍마다 해밍 거리가 max_dist 이하인 사진 쌍을 모두 찾는다."""
    ok = crops[crops["dhash"].fillna("").str.len() == 16]
    found = []
    for s1, s2 in pairs:
        a, b = ok[ok["split"] == s1], ok[ok["split"] == s2]
        if a.empty or b.empty:
            continue
        ha, hb = _to_u64(a["dhash"]), _to_u64(b["dhash"])
        for i in range(0, len(ha), chunk):
            d = _hamming(ha[i:i + chunk], hb)
            ii, jj = np.nonzero(d <= max_dist)
            for x, y in zip(ii, jj):
                ra, rb = a.iloc[i + x], b.iloc[y]
                found.append({"split_a": s1, "object_a": ra["object_id"], "path_a": ra["path"], "class_a": ra["class17"],
                              "split_b": s2, "object_b": rb["object_id"], "path_b": rb["path"], "class_b": rb["class17"],
                              "dist": int(d[x, y])})
    return pd.DataFrame(found, columns=["split_a", "object_a", "path_a", "class_a",
                                        "split_b", "object_b", "path_b", "class_b", "dist"])


def add_pixel_diff(dups: pd.DataFrame, crops_dir, size: int = 32) -> pd.DataFrame:
    """후보 쌍마다 실제 색을 비교한 차이(pix_diff)를 붙인다.

    두 사진을 size×size 컬러로 줄인 뒤 같은 위치 픽셀의 RGB 값 차이를 평균낸다(0~255).
    거의 같은 사진이면 몇 이하로 작고, 우연히 지문만 비슷한 다른 물건이면 훨씬 크다.
    dHash(흑백 64비트)만으로는 '어두운 배경 + 가운데 밝은 물건' 사진이 모두 비슷하게 나와서 추가했다.
    """
    import io
    import zipfile
    from pathlib import Path
    from PIL import Image

    if dups.empty:
        return dups.assign(pix_diff=pd.Series(dtype=float))
    need = set(zip(dups["split_a"], dups["path_a"])) | set(zip(dups["split_b"], dups["path_b"]))
    thumbs = {}
    for split in sorted({s for s, _ in need}):
        with zipfile.ZipFile(Path(crops_dir) / f"{split}.zip") as z:
            for s, p in sorted(k for k in need if k[0] == split):
                im = Image.open(io.BytesIO(z.read(p))).convert("RGB").resize((size, size), Image.BILINEAR)
                thumbs[(s, p)] = np.asarray(im, dtype=np.int16)
    diff = [float(np.abs(thumbs[(a, pa)] - thumbs[(b, pb)]).mean())
            for a, pa, b, pb in zip(dups["split_a"], dups["path_a"], dups["split_b"], dups["path_b"])]
    return dups.assign(pix_diff=np.round(diff, 1))

In [ ]:
importlib.reload(cd)
dups2 = cd.add_pixel_diff(dups, CROPS)
dups2.to_csv(CROPS / "cross_split_dups.csv", index=False)

print("색 차이 분포:", dups2["pix_diff"].describe().round(1).to_dict())
for t in [3, 5, 8, 10, 15]:
    sub = dups2[dups2["pix_diff"] <= t]
    print(f"색 차이 {t} 이하: {len(sub):,}쌍", sub.groupby(["split_a", "split_b"]).size().to_dict())

def show_pairs(df, title):
    if df.empty:
        print(title, "→ 없음"); return
    fig, axes = plt.subplots(len(df), 2, figsize=(4, 2 * len(df)))
    axes = np.atleast_2d(axes)
    for k, (_, r) in enumerate(df.iterrows()):
        axes[k, 0].imshow(img(r["split_a"], r["path_a"])); axes[k, 1].imshow(img(r["split_b"], r["path_b"]))
        axes[k, 0].set_title(f"{r['split_a']}  색차이={r['pix_diff']}", fontsize=7); axes[k, 1].set_title(r["split_b"], fontsize=7)
        for a in axes[k]:
            a.axis("off")
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()

show_pairs(dups2.sort_values("pix_diff").head(8), "색 차이가 가장 작은 8쌍")
show_pairs(dups2[dups2["pix_diff"].between(8, 12)].head(6), "색 차이 8~12 사이 6쌍 (경계 확인용)")

In [ ]:
TH = 8
RANK = {"train": 0, "val": 1, "test": 2, "phone_train": 0, "phone_tune": 1, "phone_final": 2}
close = dups2[dups2["pix_diff"] <= TH]
drop = {r["object_a"] if RANK[r["split_a"]] < RANK[r["split_b"]] else r["object_b"] for _, r in close.iterrows()}

final = crops[crops["path"].fillna("") != ""].copy()
final["dup_removed"] = final["object_id"].isin(drop)
final.to_csv(CROPS / "crops_final.csv.gz", index=False)

use = final[~final["dup_removed"]]
print(f"중복으로 뺀 물체: {len(drop)}개", final[final["dup_removed"]].groupby("split").size().to_dict())
print(f"최종 사용 물체: {len(use):,}개")
print(pd.crosstab(use["class17"], use["split"], margins=True, margins_name="합계").to_string())

## 5. GitHub에 올리기 (올리고 싶을 때만)
이 노트북과 1·2번에서 만든 파일을 `preprocess` 브랜치에 올립니다. Drive의 데이터는 올라가지 않습니다.

런타임이 새로 시작됐다면 **0 → 1 → 2번 셀을 먼저 실행**한 뒤 이 셀을 실행합니다.

In [ ]:
MESSAGE = "preprocess: 라벨 정리·분할·자르기·중복 검사 코드 추가 (전처리 1~5단계)"
colab.push(MESSAGE)